In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
print("Imported!")

Imported!


In [3]:
book_scraper_url = "https://books.toscrape.com/catalogue/category/books"
genre_to_scrape = [
    "travel_2",
    "mystery_3",
    "historical-fiction_4",
    "sequential-art_5",
    "classics_6"
]

def scrape_books_for_genre(html: str, genre: str):
    try:
        soup = BeautifulSoup(html, "html.parser")
    except Exception as e:
        print(f"Error parsing HTML for genre {genre}: {e}")
        return []
    all_books = soup.find_all("article", class_="product_pod")
    book_jsons = []
    for book in all_books:
        book_json = {
            "title": book.h3.a["title"],
            "category": genre,
            "price_in_gbp": book.find("p", class_="price_color").text.strip(),
            "star_rating": book.find("p", class_="star-rating").get("class", [])[-1],
            "availability": book.find("p", class_="availability").text.strip()
        }
        book_jsons.append(book_json)
    return book_jsons

def scrape_all_books():
    books = []
    for genre in genre_to_scrape:
        genre_url = f"{book_scraper_url}/{genre}/index.html"
        print(f"Scraping genre: {genre}")
        response = requests.request("GET", genre_url)
        if response.status_code == 200:
            books.extend(scrape_books_for_genre(response.text, genre))
    print(f"Total books scraped: {len(books)}")
    return books

print("###################################### TASK 1 ######################################")
books = scrape_all_books()
df = pd.DataFrame(books)
print(df.head(10))

###################################### TASK 1 ######################################
Scraping genre: travel_2
Scraping genre: mystery_3
Scraping genre: historical-fiction_4
Scraping genre: sequential-art_5
Scraping genre: classics_6
Total books scraped: 90
                                               title  category price_in_gbp  \
0                            It's Only the Himalayas  travel_2      Â£45.17   
1  Full Moon over Noahâs Ark: An Odyssey to Mou...  travel_2      Â£49.43   
2  See America: A Celebration of Our National Par...  travel_2      Â£48.87   
3  Vagabonding: An Uncommon Guide to the Art of L...  travel_2      Â£36.94   
4                               Under the Tuscan Sun  travel_2      Â£37.33   
5                                 A Summer In Europe  travel_2      Â£44.34   
6                           The Great Railway Bazaar  travel_2      Â£30.54   
7                   A Year in Provence (Provence #1)  travel_2      Â£56.88   
8  The Road to Little Dribbling:

In [4]:
def clean_data(df):
  star_rating = {
    "One":1,
    "Two":2,
    "Three":3,
    "Four":4,
    "Five":5
  }
  #removing numbers at the end of category column
  df['category'] = df['category'].str.replace(r'_\d+','', regex=True)
  #Converting its dtype,word-to-number and filling null values with median
  df["star_rating"] = df["star_rating"].map(star_rating)
  df['star_rating'] = df['star_rating'].fillna(df['star_rating'].median())
  # Removing symbols in price column and filling null values with mean
  df["price_in_gbp"] = df["price_in_gbp"].str.replace("Â£", "").astype(float, errors='raise')
  df["price_in_gbp"] = df["price_in_gbp"].fillna(df["price_in_gbp"].mean())
  # converting availability column to bool
  df['availability'] = df['availability'].str.contains("In stock", case="False", na=False)
  return df
print("###################################### TASK 2 ######################################")
df = clean_data(df)
print(df.head())

###################################### TASK 2 ######################################
                                               title category  price_in_gbp  \
0                            It's Only the Himalayas   travel         45.17   
1  Full Moon over Noahâs Ark: An Odyssey to Mou...   travel         49.43   
2  See America: A Celebration of Our National Par...   travel         48.87   
3  Vagabonding: An Uncommon Guide to the Art of L...   travel         36.94   
4                               Under the Tuscan Sun   travel         37.33   

   star_rating  availability  
0            2          True  
1            4          True  
2            3          True  
3            2          True  
4            3          True  


In [5]:
def gbp_to_inr(books):
  to_inr = 105.50
  df['price_in_inr'] = df['price_in_gbp'] * to_inr
  return df
print("###################################### TASK 3 ######################################")
df = gbp_to_inr(books)
print(df.head(10))

###################################### TASK 3 ######################################
                                               title category  price_in_gbp  \
0                            It's Only the Himalayas   travel         45.17   
1  Full Moon over Noahâs Ark: An Odyssey to Mou...   travel         49.43   
2  See America: A Celebration of Our National Par...   travel         48.87   
3  Vagabonding: An Uncommon Guide to the Art of L...   travel         36.94   
4                               Under the Tuscan Sun   travel         37.33   
5                                 A Summer In Europe   travel         44.34   
6                           The Great Railway Bazaar   travel         30.54   
7                   A Year in Provence (Provence #1)   travel         56.88   
8  The Road to Little Dribbling: Adventures of an...   travel         23.21   
9          Neither Here nor There: Travels in Europe   travel         38.95   

   star_rating  availability  price_in_inr  


In [9]:
def database():
  connection = sqlite3.connect("books.db")
  cursor = connection.cursor()
  cursor.execute("DROP TABLE IF EXISTS books")
  cursor.execute("DROP TABLE IF EXISTS categories")
  print("Creating categories table")
  cursor.execute("""
             CREATE TABLE IF NOT EXISTS categories(
             category_id INTEGER PRIMARY KEY AUTOINCREMENT,
             category_name TEXT NOT NULL UNIQUE)
             """)
  print('Table 1 created')
  print("Creating books table")
  cursor.execute("""
                CREATE TABLE IF NOT EXISTS books(
                book_id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT NOT NULL,
                price_in_gbp REAL NOT NULL,
                price_in_inr REAL NOT NULL,
                rating INTEGER NOT NULL,
                availability BOOL NOT NULL,
                category_id INTEGER NOT NULL,
                FOREIGN KEY(category_id) REFERENCES categories(category_id))
                """)
  print('Table 2 created')
  print("Database Created Successfully")
  return cursor
print("###################################### TASK 4 ######################################")
cursor = database()
print("Inserting values to the table")
def category(cursor, books):
  inserted_categories=[]
  unique_categories = books['category'].unique()
  for i, category in enumerate(unique_categories):
    inserted_categories.append((i, category))
  cursor.execute("DELETE FROM categories")
  cursor.executemany("INSERT INTO categories(category_id,category_name) VALUES (?,?)", inserted_categories)
  print(f"{len(inserted_categories)} categories inserted")
category(cursor, df)
def insert_books(cursor, books):
  categories = {
      "travel": 0,
      "mystery": 1,
      "historical-fiction": 2,
      "sequential-art": 3,
      "classics": 4
  }
  cursor.execute("DELETE FROM books")
  query="""
    INSERT INTO books(
      title,
      price_in_gbp,
      price_in_inr,
      rating,
      availability,
      category_id
    ) VALUES (?,?,?,?,?,?)"""
  book_data=[
      (
          book["title"],
          book["price_in_gbp"],
          book["price_in_inr"],
          book["star_rating"],
          book["availability"],
          categories[book["category"]]
      )
      for i, book in books.iterrows()
  ]
  cursor.executemany(query, book_data)
  print(f"{len(book_data)} books inserted")
insert_books(cursor, df)


###################################### TASK 4 ######################################
Creating categories table
Table 1 created
Creating books table
Table 2 created
Database Created Successfully
Inserting values to the table
5 categories inserted
90 books inserted


In [11]:
# Applying queries
print("Query-1")
query1 = """SELECT title FROM books WHERE rating = 5"""
result1 = cursor.execute(query1).fetchall()
print(result1)
print(f"There are {len(result1)} books with 5-star rating")
print("-" * 10)
print("Query-2")
query2 = """SELECT title,price_in_gbp FROM books ORDER BY price_in_gbp DESC LIMIT 1"""
result2=cursor.execute(query2).fetchall()
print(f"The most expensive book is :")
print(result2)
print("-" * 10)
print("Query-3")
query3="""
SELECT DISTINCT
categories.category_name,title
FROM books
JOIN categories
ON books.category_id=
categories.category_id
ORDER BY books.price_in_gbp DESC
LIMIT 5 """
result3=cursor.execute(query3).fetchall()
print("Categories and names of top-5 expensive books")
print(result3)
print("-" * 10)
print("Query-4")
query4="""SELECT title,price_in_gbp FROM books WHERE price_in_gbp BETWEEN 10 AND 20"""
result4=cursor.execute(query4).fetchall()
print(f"There are {len(result4)} books with prices between 10 and 20 GBP")
print(result4)
print("-" * 10)
print("Query-5")
query5="""SELECT books.title,
          books.price_in_gbp
         FROM books
      JOIN categories
      ON books.category_id =
      categories.category_id
      WHERE categories.category_name =
      'mystery'
      """
result5 = cursor.execute(query5).fetchall()
print(f"There are {(len(result5))} books from mystery category")
print(result5)
 

Query-1
[('1,000 Places to See Before You Die',), ('A Time of Torment (Charlie Parker #14)',), ('What Happened on Beale Street (Secrets of the South Mysteries #2)',), ("The Bachelor Girl's Guide to Murder (Herringford and Watts Mysteries #1)",), ('A Flight of Arrows (The Pathfinders #2)',), ('Mrs. Houdini',), ('The Passion of Dolssa',), ('Voyager (Outlander #3)',), ('The Red Tent',), ("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)",), ('Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Princess Jellyfish 2-in-1 Omnibus #1)',)]
There are 11 books with 5-star rating
----------
Query-2
The most expensive book is :
[('Boar Island (Anna Pigeon #19)', 59.48)]
----------
Query-3
Categories and names of top-5 expensive books
[('mystery', 'Boar Island (Anna Pigeon #19)'), ('classics', 'Candide'), ('classics', 'Animal Farm'), ('travel', 'A Year in Provence (Provence #1)'), ('mystery', 'The Past Never Ends')]
----------
Query-4
There are 20 books with prices between 10 and 20 GBP
[('In a Dark

In [12]:
df_books = pd.read_sql("SELECT * FROM books", cursor.connection)
df_categories = pd.read_sql("SELECT * FROM categories", cursor.connection)
query_join = """
  SELECT books.title, categories.category_name,price_in_gbp
  FROM books
  JOIN categories ON books.category_id = categories.category_id
"""
df_sql_join = pd.read_sql(query_join, cursor.connection)
df_pandas_merge = pd.merge(df_books, df_categories, on = "category_id")
print("sql join result")
print(df_sql_join.head())
print("pandas merge result")
print(df_pandas_merge.head())

sql join result
                                               title category_name  \
0                            It's Only the Himalayas        travel   
1  Full Moon over Noahâs Ark: An Odyssey to Mou...        travel   
2  See America: A Celebration of Our National Par...        travel   
3  Vagabonding: An Uncommon Guide to the Art of L...        travel   
4                               Under the Tuscan Sun        travel   

   price_in_gbp  
0         45.17  
1         49.43  
2         48.87  
3         36.94  
4         37.33  
pandas merge result
   book_id                                              title  price_in_gbp  \
0        1                            It's Only the Himalayas         45.17   
1        2  Full Moon over Noahâs Ark: An Odyssey to Mou...         49.43   
2        3  See America: A Celebration of Our National Par...         48.87   
3        4  Vagabonding: An Uncommon Guide to the Art of L...         36.94   
4        5                              